# 02 — Data Collection
Goal: pull all available match and player data from the API, flatten it, and save as parquet files under `data/processed/`.

In [ ]:
import sys
sys.path.insert(0, '..')

import datetime
import pandas as pd
from pathlib import Path
from src.api.client import PadelAPIClient
from src.processing.flatten import flatten_match, flatten_player, flatten_tournament

client = PadelAPIClient(requests_per_minute=10)
PROCESSED_DIR = Path('../data/processed')
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

today = datetime.date.today().isoformat()
print(f'Collecting data up to {today}')

## 1. Players
Fetch all players across all pages.

In [ ]:
raw_players = client.get_all_players()
print(f'Fetched {len(raw_players)} players')

df_players = pd.DataFrame([flatten_player(p) for p in raw_players])
df_players['birthdate'] = pd.to_datetime(df_players['birthdate'], errors='coerce')
print(df_players.shape)
df_players.head()

## 2. Matches
Fetch all completed matches (paginated). The free tier covers the last 6 months.

In [ ]:
raw_matches = client.get_all_matches(before_date=today)
print(f'Fetched {len(raw_matches)} matches total')

finished = [m for m in raw_matches if m.get('status') == 'finished']
print(f'Finished matches: {len(finished)}')

df_matches = pd.DataFrame([flatten_match(m) for m in finished])
df_matches['played_at'] = pd.to_datetime(df_matches['played_at'], errors='coerce')
print(df_matches.shape)
df_matches.head()

## 3. Tournaments
Fetch metadata for every tournament referenced in the match data.

In [ ]:
tournament_ids = df_matches['tournament_id'].dropna().astype(int).unique()
print(f'Unique tournaments in match data: {len(tournament_ids)}')

raw_tournaments = [client.get_tournament(int(tid)) for tid in tournament_ids]
df_tournaments = pd.DataFrame([flatten_tournament(t) for t in raw_tournaments])
df_tournaments['start_date'] = pd.to_datetime(df_tournaments['start_date'], errors='coerce')
df_tournaments['end_date'] = pd.to_datetime(df_tournaments['end_date'], errors='coerce')
print(df_tournaments.shape)
df_tournaments.head()

## 4. Save to Parquet

In [ ]:
df_players.to_parquet(PROCESSED_DIR / 'players.parquet', index=False)
df_matches.to_parquet(PROCESSED_DIR / 'matches.parquet', index=False)
df_tournaments.to_parquet(PROCESSED_DIR / 'tournaments.parquet', index=False)
print('Saved:')
print(f'  players.parquet    {len(df_players):>5} rows')
print(f'  matches.parquet    {len(df_matches):>5} rows')
print(f'  tournaments.parquet {len(df_tournaments):>4} rows')

## 5. Dataset Summary

In [ ]:
# Date range
print('Match date range:', df_matches['played_at'].min().date(), '→', df_matches['played_at'].max().date())
print()

# Matches by category
print('Matches by category:')
print(df_matches['category'].value_counts().to_string())
print()

# Matches by round
print('Matches by round:')
print(df_matches[['round', 'round_name']].value_counts().sort_index().to_string())
print()

# Null rates for key columns
key_cols = ['winner', 'sets_won_t1', 'sets_won_t2', 't1_p1', 't1_p2', 't2_p1', 't2_p2']
null_pct = df_matches[key_cols].isna().mean().mul(100).round(1)
print('Null % for key columns:')
print(null_pct.to_string())